# Completeness of the low-level feature representation

Recruitment of *informative* low-level (**L**) neurons into labelled binding circuits, graded by
PNG F1, for **N3P2 / ALL** and **N4P2 / ALL** (three detection trials each) at the post-trained
checkpoint. For a PNG anchored at layer *l* the L neuron lives in layer *l*-1, so the coverage
denominator at anchor layer *l* is the informative layer-(*l*-1) neurons. Anchor layer 1 is
excluded (its L neuron sits in the fixed Poisson input layer); the figure plots the final anchor
layer (4).

**Dependencies:**

Build the canonical HFB annotation tables for all detection trials of both datasets first:
```bash
./scripts/analysis/build_annotation_table.py ./experiments/n3p2/train_n3p2_lrate_0_04_181023 ALL --all-detection -v
./scripts/analysis/build_annotation_table.py ./experiments/n4p2/train_n4p2_lrate_0_02_181023 ALL --all-detection -v
```

**Plots:**

- Completeness funnel - per-label nested recruitment (informative pool -> structural -> F1 > 0.5
  -> F1 > 0.7 -> F1 > 0.9), two vertically stacked sub-panels (N3P2, N4P2), log y-axis, labels
  clustered by conformation (convex then concave). Structural participation is broad while the
  high-F1 functional subset is sparse and convex-biased.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hsnn import viz
from hsnn.pipeline import reuse
from hsnn.pipeline.reuse import completeness
from hsnn.utils import handler, io

viz.setup_journal_env()

In [ ]:
# === CONFIGURATION ===
# Both N*/ALL combinations; per-dataset detection trials and labels are resolved from metadata.
DATASETS = {
    "N3P2": "n3p2/train_n3p2_lrate_0_04_181023",
    "N4P2": "n4p2/train_n4p2_lrate_0_02_181023",
}
MODEL_TYPE = "ALL"                                # FF + LAT + FB architecture
CHECKPOINT = -1                                   # post-trained (last) state
ANCHOR_LAYERS = (2, 3, 4)                           # layer 1 excluded (L is Poisson input)
PANEL_LAYER = 4                                    # figure shows the final anchor layer only
INFO_BITS = 2 / 3                                  # manuscript informativeness threshold
INFO_THRESHOLDS = (2 / 3,)                          # informativeness tau on info_bits

# Recruitment tiers (reused from the completeness module): structural (any labelled PNG) then
# successively stricter F1 cuts; each is a subset of the looser one, so per-label bars nest.
TIERS = completeness.DEFAULT_TIERS
TIER_ORDER = tuple(name for name, _ in TIERS)
LEVEL_ORDER = ("pool", *TIER_ORDER)
LEVEL_LABEL = {
    "pool": "Informative pool", "structural": "Structural",
    "f1>0.5": "F1 > 0.5", "f1>0.7": "F1 > 0.7", "f1>0.9": "F1 > 0.9",
}
CMAP = "Blues"
RAMP_STOPS = (0.25, 0.95)  # lightest / darkest sample positions along the colormap

OUTPUT_DIR = io.BASE_DIR / "out/figures/supplementary/fig_S9"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load both datasets (three detection trials each)

For each dataset the persisted annotation tables (`hfb_annotations`, `neuron_information`) are
read for every detection trial; nothing is rebuilt. Detection trials and feature labels are
resolved from the experiment metadata. Labels are clustered by conformation (convex, then
concave) for the figure x-axis.

In [ ]:
tables: dict[str, dict[str, dict[str, pd.DataFrame]]] = {}
DETECTION_TRIALS: dict[str, list[str]] = {}
ALL_LABELS: dict[str, list[str]] = {}
CONVEX: dict[str, list[str]] = {}
CONCAVE: dict[str, list[str]] = {}
PANEL_ORDER: dict[str, list[str]] = {}

for ds, experiment in DATASETS.items():
    expt = handler.ExperimentHandler(experiment)
    DETECTION_TRIALS[ds] = expt.metadata.get_trials_dict(MODEL_TYPE)["detection"]
    tables[ds] = {
        t: reuse.load_tables(
            reuse.open_store(experiment, MODEL_TYPE, trial=t, checkpoint=CHECKPOINT))
        for t in DETECTION_TRIALS[ds]
    }
    rep_info = tables[ds][DETECTION_TRIALS[ds][0]]["neuron_information"]
    labels = sorted(completeness.feature_label(rep_info).unique())
    ALL_LABELS[ds] = labels
    CONVEX[ds] = [lab for lab in labels if completeness.conformation_of(lab) == "convex"]
    CONCAVE[ds] = [lab for lab in labels if completeness.conformation_of(lab) == "concave"]
    PANEL_ORDER[ds] = CONVEX[ds] + CONCAVE[ds]

    print(f"[{ds}] {experiment}")
    print(f"  detection trials: {DETECTION_TRIALS[ds]}")
    print(f"  feature labels ({len(labels)}): {labels}")
    for t, tab in tables[ds].items():
        ann = tab["hfb_annotations"]
        print(f"    {t}: {len(ann):>6d} significant PNGs "
              f"({int(ann['png_pref_label'].notna().sum())} labelled)")

## Recruitment counting and three-trial aggregate

For each dataset x trial the completeness module counts, per anchor layer / informativeness
threshold / recruitment tier, the informative-pool size (the denominator) and the reuse-collapsed
neurons recruited at that tier. Counts and pool sizes are reported as the per-trial mean across
the three detection trials.

In [ ]:
counts_frames = []
for ds in DATASETS:
    for t in DETECTION_TRIALS[ds]:
        res = completeness.recruitment_tables(
            tables[ds][t]["hfb_annotations"],
            tables[ds][t]["neuron_information"],
            ALL_LABELS[ds],
            anchor_layers=ANCHOR_LAYERS,
            info_thresholds=INFO_THRESHOLDS,
            tiers=TIERS,
        )
        counts_frames.append(res.counts.assign(dataset=ds, trial_id=t))
counts = pd.concat(counts_frames, ignore_index=True)


def _sd(x: pd.Series) -> float:
    """Sample standard deviation (the across-trial spread)."""
    return float(np.std(x, ddof=1))


# Per-label / tier aggregate at every anchor layer (drives the funnel and the backing CSV).
label_tier = (
    counts.groupby(["dataset", "anchor_layer", "threshold", "label", "conformation", "tier"])
    .agg(pool_mean=("pool", "mean"), pool_sd=("pool", _sd),
         any_count_mean=("recruited_any", "mean"), any_count_sd=("recruited_any", _sd),
         matched_count_mean=("recruited_matched", "mean"), matched_count_sd=("recruited_matched", _sd))
    .reset_index()
)
label_tier["frac_any_mean"] = label_tier["any_count_mean"] / label_tier["pool_mean"].where(
    label_tier["pool_mean"] > 0)
label_tier.to_csv(OUTPUT_DIR / "s9_completeness_funnel_data.csv", index=False)
print(f"Wrote s9_completeness_funnel_data.csv ({len(label_tier)} rows)")

## The completeness funnel (anchor layer 4, both datasets)

Two vertically stacked sub-panels (N3P2 top, N4P2 bottom) at anchor layer 4. Each is a per-label
bar chart, labels clustered by conformation (convex then concave). For each label, nested bars in
a sequential colour ramp form a funnel: the broadest (lightest) bar is the informative pool, then
structural recruitment, then F1 > 0.5, > 0.7, > 0.9 (darkening to the strict tip). The log y-axis
keeps the small high-F1 counts and their convex skew visible; the pool count is annotated above
each group.

In [ ]:
CLUSTER_GAP = 0.9


def _cluster_positions(n_convex: int, n_concave: int, gap: float = CLUSTER_GAP) -> np.ndarray:
    """x positions for a convex cluster then a concave cluster, with a visible gap."""
    left = np.arange(n_convex, dtype=float)
    right = np.arange(n_concave, dtype=float) + n_convex + gap
    return np.concatenate([left, right])


def _short(label: str) -> str:
    """Compact tick label, e.g. 'bottom-convex' -> 'bottom'."""
    return label.rsplit("-", 1)[0]


def _level_colours(cmap: str = CMAP, stops: tuple = RAMP_STOPS) -> dict:
    """Maps each funnel level to a colour sampled along ``cmap`` (lightest pool -> darkest tip)."""
    ramp = plt.get_cmap(cmap)
    xs = np.linspace(stops[0], stops[1], len(LEVEL_ORDER))
    return {lev: ramp(float(x)) for lev, x in zip(LEVEL_ORDER, xs)}


def funnel_panel(ax: plt.Axes, dataset: str, colours: dict) -> None:
    """Draws one dataset's nested pool -> tier recruitment funnel at the panel layer."""
    order = PANEL_ORDER[dataset]
    n_cx, n_cc = len(CONVEX[dataset]), len(CONCAVE[dataset])
    xpos = _cluster_positions(n_cx, n_cc)
    sub = label_tier[
        (label_tier["dataset"] == dataset)
        & (label_tier["anchor_layer"] == PANEL_LAYER)
        & np.isclose(label_tier["threshold"], 2 / 3)
    ]
    tiers_wide = sub.pivot_table(index="label", columns="tier", values="matched_count_mean").reindex(order)
    pool = sub.groupby("label")["pool_mean"].first().reindex(order)
    level_values = {"pool": pool, **{tier: tiers_wide[tier] for tier in TIER_ORDER}}

    for z, level in enumerate(LEVEL_ORDER):
        h = np.asarray(level_values[level], dtype=float)
        h[h <= 0] = np.nan  # absent bar = uncovered at this level (clean on a log axis)
        label = LEVEL_LABEL[level]
        if LEVEL_LABEL[level] == "Structural":
            label = "F1 > 0"
        elif LEVEL_LABEL[level] == "Informative pool":
            label = "Informative"
        ax.bar(xpos, h, width=0.8, color=colours[level], zorder=z + 1, label=label)

    ax.set_yscale("log")
    ax.set_ylim(bottom=0.7)
    pool_h = np.asarray(pool, dtype=float)
    for x, nn, ph in zip(xpos, pool, pool_h):
        y = (ph if ph and not np.isnan(ph) else 0.8) * 1.35
        ax.text(x, y, f"{nn:.0f}", ha="center", va="bottom", fontsize="x-small", color="0.4")

    ax.set_xticks(xpos)
    ax.set_xticklabels([_short(lab).capitalize() for lab in order], rotation=45, ha="right")
    ax.set_title(dataset, fontweight="bold", loc="left", pad=16)
    ax.set_ylabel("Informative L neurons")
    cx_centre = (n_cx - 1) / 2
    cc_centre = n_cx + CLUSTER_GAP + (n_cc - 1) / 2
    for centre, name in ((cx_centre, "Convex"), (cc_centre, "Concave")):
        ax.text(centre, -0.30, name, ha="center", va="top", fontsize="small", color="black",
                fontweight="bold", transform=ax.get_xaxis_transform())
    ax.spines[["top", "right"]].set_visible(False)


colours = _level_colours()
ds_order = list(DATASETS)
fig, axes = plt.subplots(len(ds_order), 1, figsize=(5.5, 5), sharey=True)
axes = np.atleast_1d(axes)
for ax, ds in zip(axes, ds_order):
    funnel_panel(ax, ds, colours)
handles, leg_labels = axes[0].get_legend_handles_labels()
axes[0].legend(handles, leg_labels, frameon=True, fontsize="small", bbox_to_anchor=(1.02, 1.1))
fig.tight_layout()
viz.save_figure(fig, OUTPUT_DIR / "s9_completeness_funnel.pdf", overwrite=False,
                bbox_inches="tight")
plt.show()